# minLSTM vs LSTM for Predictive Beamforming

Does [minLSTM](https://arxiv.org/abs/2410.01201) train effectively on trajectory
prediction, and does it buy anything over the LSTM this project already uses?

This notebook answers three separate questions, which are easy to conflate:

| Question | Where | Honest answer requires |
|---|---|---|
| Is a learned model worth it **at all**? | §4 | Tuned classical baselines, not a strawman |
| Is minLSTM **faster**? | §5 | A sequence-length sweep, not a single point |
| Is minLSTM **better**? | §6–7 | Equal search budget, multiple seeds, held-out trajectories |

### What changed from notebooks 1–4, and why

Running the old setup unchanged would have produced a confident but meaningless
comparison. Four things had to be fixed first:

1. **There was no non-learned baseline.** On the three original trajectories a
   zero-parameter least-squares fit reaches 0.062 m where the LSTM reports
   0.107 m, and two-point extrapolation reaches 0.032 m on the sharp turn where
   the LSTM reports 0.104 m. The published results never established that an RNN
   was needed.
2. **The validation split leaked.** The tail-20% split put window 152 in
   validation and window 151 in training — they share 9 of 10 positions. We now
   hold out whole trajectories.
3. **The model saw exact coordinates.** No base station has those. Inputs are now
   observed through a sensing-aided positioning error (σ ≈ 0.5 m) while targets
   remain the true positions, which is what makes filtering non-trivial.
4. **The motion was too easy.** Smooth hand-drawn paths are tracked almost
   perfectly by arithmetic. The corpus here uses maneuvering coordinated-turn
   motion, the classic case a constant-velocity filter cannot follow.

Supporting code lives in `src/` so the scan can be unit-tested and the search is
reproducible. This notebook is the driver.

## 1. Environment

The paper's Figure 1 timings were measured on a T4, so the runtime curves in §5 are directly comparable if this reports one.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || echo "no GPU detected - §5 timings will not be meaningful"

In [ ]:
!pip install -q optuna

In [ ]:
import sys, pathlib, platform, time, json, warnings
import numpy as np, torch, matplotlib, matplotlib.pyplot as plt

# src/ lives one level up from notebooks/
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.train import resolve_device
DEVICE = resolve_device()

print(f"python      {platform.python_version()}")
print(f"torch       {torch.__version__}")
print(f"numpy       {np.__version__}")
print(f"device      {DEVICE}")
if DEVICE.type == "cuda":
    print(f"gpu         {torch.cuda.get_device_name(0)}")
    print(f"cuda        {torch.version.cuda}")
print(f"project root {ROOT}")

### Run scale

`QUICK = True` runs the whole notebook end to end in a few minutes so you can
confirm nothing is broken before committing GPU hours. Set it to `False` for the
real run.

In [ ]:
QUICK = True   # <-- set False for the full study

CFG = dict(
    sigma       = 0.5,     # sensing-aided localisation error, metres
    dt          = 0.1,     # slot duration, seconds -> a 16 m/s user moves 1.6 m/slot
    steps       = 200,     # timesteps per trajectory
    seed        = 0,
)
if QUICK:
    CFG.update(n_traj=60,  n_trials=8,   n_seeds=2, max_epochs=40,  patience=8)
else:
    CFG.update(n_traj=600, n_trials=150, n_seeds=5, max_epochs=300, patience=30)

RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
MODELS  = ROOT / "models";  MODELS.mkdir(exist_ok=True)
CFG

### Plot style

Three series at most per chart, using palette slots that are validated
colourblind-safe as a set (CVD ΔE 9.2, normal-vision ΔE 24.0). Grid and axes are
kept recessive so the data carries the contrast.

In [ ]:
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED = "#0b0b0b", "#52514e"
SERIES = {"lstm": BLUE, "plain_lstm": ORANGE, "minlstm": AQUA, "minlstm_log": AQUA}
LABELS = {"lstm": "LSTM (cuDNN)", "plain_lstm": "LSTM (plain PyTorch)",
          "minlstm": "minLSTM", "minlstm_log": "minLSTM (log-space)"}

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9c8c2", "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.grid": True, "grid.color": "#dedcd5", "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 10,
    "lines.linewidth": 2.0, "lines.markersize": 5,
    "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
})
print("style set")

## 2. Correctness tests

The parallel scan is the one component that fails silently — a wrong scan still
produces plausible-looking numbers. These tests assert that the scan reproduces
the unrolled recurrence exactly, that log-space survives 4096 steps where the
vanilla path underflows, and that the plain-PyTorch LSTM computes the same
function as cuDNN's (without which the §5 timing comparison would be measuring
two different models).

In [ ]:
!cd {ROOT} && python -m pytest tests/ -q

## 3. Trajectory corpus

Three properties make this corpus harder than the original hand-drawn paths, and
each is deliberate:

- **Maneuvering motion.** The turn rate is piecewise constant and switches at
  random times. A constant-velocity filter is right between maneuvers and wrong
  at every switch — the headroom a learned model has to exploit.
- **Noisy observations.** Inputs carry σ = 0.5 m of positioning error; targets
  are the true grid positions. The base station must *estimate*, not read off.
- **Mobility regimes.** Pedestrian / cyclist / vehicular bands, so a model can be
  trained on some and tested on a kind of motion it has never seen.

In [ ]:
from src.trajectories import generate_corpus, REGIMES
from src.data import build_dataset, split_by_trajectory, split_by_regime

rng = np.random.default_rng(CFG["seed"])
corpus = generate_corpus(n=CFG["n_traj"], steps=CFG["steps"], rng=rng,
                         sigma=CFG["sigma"], dt=CFG["dt"])
train_c, val_c, test_c = split_by_trajectory(corpus, (0.7, 0.15, 0.15), rng=rng)

from collections import Counter
print(f"{len(train_c)} train / {len(val_c)} val / {len(test_c)} test trajectories")
print("regimes:", dict(Counter(t['regime'] for t in corpus)))
print(f"speeds : {min(t['speed'] for t in corpus):.1f} - {max(t['speed'] for t in corpus):.1f} m/s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, regime in zip(axes, REGIMES):
    shown = [t for t in corpus if t["regime"] == regime][:6]
    for t in shown:
        ax.plot(t["true"][:, 0], t["true"][:, 1], color=BLUE, lw=1.4, alpha=0.75)
    if shown:   # one trajectory's noisy observations, to show what the model sees
        ax.plot(shown[0]["observed"][:, 0], shown[0]["observed"][:, 1],
                color=ORANGE, lw=0, marker=".", ms=2.5, alpha=0.65)
    ax.set_title(f"{regime}  ({REGIMES[regime]['speed'][0]:.0f}-{REGIMES[regime]['speed'][1]:.0f} m/s)")
    ax.set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
axes[0].plot([], [], color=BLUE, lw=1.4, label="true path")
axes[0].plot([], [], color=ORANGE, lw=0, marker=".", ms=5, label=f"observed (σ={CFG['sigma']} m)")
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Corpus trajectories in the O1_60 corridor", y=1.02)
plt.savefig(RESULTS / "corpus_trajectories.png"); plt.show()

## 4. The bar to beat

Before comparing architectures, establish what *no* learned model is needed to
achieve. Both classical predictors get their hyperparameters tuned with the same
seriousness as the networks — an untuned baseline is a strawman, and reviewers
in the tracking community will say so.

The window length is the interesting knob. Longer windows average away more
positioning noise but lag harder through a maneuver, so there is a genuine
optimum rather than a monotone trend.

In [ ]:
from src.baselines import BASELINES, kalman_cv, ls_linear

def baseline_mae(trajs, W, fn):
    errs = [np.linalg.norm(fn(t["observed"][i:i+W]) - t["true"][i+W])
            for t in trajs for i in range(len(t["observed"]) - W)]
    return float(np.mean(errs))

WINDOWS = [3, 5, 10, 20, 40, 80]
sweep = {name: [baseline_mae(val_c, W, fn) for W in WINDOWS]
         for name, fn in BASELINES.items()}
sweep["kalman"] = [baseline_mae(val_c, W, kalman_cv(dt=CFG["dt"], sigma=CFG["sigma"], q=1.0))
                   for W in WINDOWS]

for name, vals in sweep.items():
    best = int(np.argmin(vals))
    print(f"{name:<18} best {vals[best]:.3f} m at W={WINDOWS[best]}")

In [ ]:
# Tune the Kalman's process noise at its best window: q trades responsiveness
# against smoothing and is the filter's real hyperparameter.
W_STAR = WINDOWS[int(np.argmin(sweep["kalman"]))]
q_grid = [1e-2, 1e-1, 1e0, 1e1, 1e2]
q_mae = [baseline_mae(val_c, W_STAR, kalman_cv(dt=CFG["dt"], sigma=CFG["sigma"], q=q))
         for q in q_grid]
Q_STAR = q_grid[int(np.argmin(q_mae))]

CLASSICAL_BAR = baseline_mae(test_c, W_STAR, kalman_cv(dt=CFG["dt"], sigma=CFG["sigma"], q=Q_STAR))
CLASSICAL_LS  = baseline_mae(test_c, W_STAR, ls_linear)
print(f"tuned Kalman : q={Q_STAR:g}, W={W_STAR}  ->  {CLASSICAL_BAR:.3f} m on test")
print(f"tuned LS fit : W={W_STAR}             ->  {CLASSICAL_LS:.3f} m on test")
print(f"\nTarget for every learned model below: beat {min(CLASSICAL_BAR, CLASSICAL_LS):.3f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for name, colour in [("ls_linear", BLUE), ("kalman", ORANGE), ("constant_velocity", AQUA)]:
    ax.plot(WINDOWS, sweep[name], marker="o", color=colour, label=name)
    lo = int(np.argmin(sweep[name]))
    ax.annotate(f"{sweep[name][lo]:.2f} m", (WINDOWS[lo], sweep[name][lo]),
                textcoords="offset points", xytext=(0, -16), ha="center",
                fontsize=9, color=MUTED)
ax.set_xscale("log"); ax.set_xticks(WINDOWS); ax.set_xticklabels(WINDOWS)
ax.set_xlabel("window length W (timesteps)")
ax.set_ylabel("MAE (m)")
ax.set_title("Classical baselines: noise averaging vs maneuver lag")
ax.legend()
plt.savefig(RESULTS / "baseline_window_sweep.png"); plt.show()

## 5. Computational study

The paper reports minGRU/minLSTM training 175×–1361× faster than their classical
counterparts at sequence lengths 512–4096. **Do not expect that at W=10.**

Two reasons the headline number will not reproduce at this project's scale:

- `nn.LSTM` dispatches to a single fused cuDNN kernel. A plain-PyTorch minLSTM
  is three linears plus a scan assembled from several elementwise ops — many
  small kernel launches. At 10 timesteps the GPU is idle and wall-clock is
  launch overhead, so minLSTM can easily come out *slower*.
- The paper itself rewrote LSTM and GRU in plain PyTorch precisely because
  comparing against cuDNN measures kernel engineering, not architecture.

So we measure all three and report the **crossover**: the sequence length past
which the scan wins. That is a real, reproducible result; a single-point speed
claim at W=10 is not.

In [ ]:
from src.models import build_model

def time_training_step(name, seq_len, batch=64, hidden=64, layers=2,
                       warmup=10, iters=30):
    """Forward + loss + backward, in milliseconds.

    CUDA is asynchronous, so every timed region is bracketed by synchronize();
    without that we would be timing kernel *launches*, not kernel execution.
    """
    model = build_model(name, hidden_size=hidden, num_layers=layers, dropout=0.0).to(DEVICE)
    opt = torch.optim.Adam(model.parameters())
    x = torch.randn(batch, seq_len, 2, device=DEVICE)
    y = torch.randn(batch, 2, device=DEVICE)
    loss_fn = torch.nn.MSELoss()

    def one_step():
        opt.zero_grad()
        loss_fn(model(x), y).backward()
        opt.step()

    for _ in range(warmup):
        one_step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    for _ in range(iters):
        one_step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / iters * 1000

    peak = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else float("nan")
    del model, opt, x, y
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return ms, peak

In [ ]:
SEQ_LENS = [10, 32, 64, 128, 256, 512] if QUICK else [10, 32, 64, 128, 256, 512, 1024, 2048, 4096]
ARCHS = ["lstm", "plain_lstm", "minlstm"]

timing = {a: {"ms": [], "mem": []} for a in ARCHS}
for T in SEQ_LENS:
    for a in ARCHS:
        ms, mem = time_training_step(a, T)
        timing[a]["ms"].append(ms); timing[a]["mem"].append(mem)
    row = "  ".join(f"{LABELS[a]}={timing[a]['ms'][-1]:7.2f}ms" for a in ARCHS)
    print(f"T={T:<5} {row}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))

for a in ARCHS:
    ax1.plot(SEQ_LENS, timing[a]["ms"], marker="o", color=SERIES[a], label=LABELS[a])
ax1.set_xscale("log", base=2); ax1.set_yscale("log")
ax1.set_xlabel("sequence length"); ax1.set_ylabel("training step (ms)")
ax1.set_title("Training step time")
ax1.legend()

# Where does the scan overtake each LSTM? Report it rather than asserting it.
mins = np.array(timing["minlstm"]["ms"])
for a, style in [("lstm", "-"), ("plain_lstm", "--")]:
    faster = np.where(mins < np.array(timing[a]["ms"]))[0]
    if len(faster):
        x = SEQ_LENS[faster[0]]
        ax1.axvline(x, color=SERIES[a], ls=":", lw=1.2, alpha=0.8)
        print(f"minLSTM overtakes {LABELS[a]} at T={x}")
    else:
        print(f"minLSTM never overtakes {LABELS[a]} within T<={SEQ_LENS[-1]}")

if DEVICE.type == "cuda":
    for a in ARCHS:
        ax2.plot(SEQ_LENS, timing[a]["mem"], marker="o", color=SERIES[a], label=LABELS[a])
    ax2.set_xscale("log", base=2)
    ax2.set_xlabel("sequence length"); ax2.set_ylabel("peak memory (MiB)")
    ax2.set_title("Peak memory — the scan's cost")
    ax2.legend()
else:
    ax2.axis("off"); ax2.text(0.5, 0.5, "memory requires CUDA", ha="center", color=MUTED)

plt.savefig(RESULTS / "timing_crossover.png"); plt.show()

### Inference latency

Training throughput is the paper's headline, but it is not the metric a base
station cares about. What matters there is the cost of one forward pass in
**sequential mode** — the beam has to be chosen inside a slot.

In [ ]:
@torch.no_grad()
def inference_latency(name, seq_len=20, batch=1, hidden=64, layers=2, iters=200):
    model = build_model(name, hidden_size=hidden, num_layers=layers, dropout=0.0).to(DEVICE).eval()
    x = torch.randn(batch, seq_len, 2, device=DEVICE)
    mode = "sequential" if name.startswith("minlstm") else None
    kw = {"mode": mode} if mode else {}
    for _ in range(20):
        model(x, **kw)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        model(x, **kw)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000

print(f"{'model':<24}{'params':>10}{'latency/slot':>15}")
for a in ARCHS:
    m = build_model(a, hidden_size=64, num_layers=2)
    print(f"{LABELS[a]:<24}{sum(p.numel() for p in m.parameters()):>10,}"
          f"{inference_latency(a):>13.3f}ms")

### 6a. Why log-space is not optional

The two scan formulations are different models, not two ways of computing one.
The vanilla scan reconstructs the recurrence as `A_t * (h_0 + cumsum(i'h̃ / A_t))`
with `A_t = cumprod(f')`, and since `f'` averages about 0.5, `A_t` halves every
step. By 20 steps it is ~1e-10 and by 80 it is ~1e-37, where fp32 gradients
become NaN. Log-space carries the same recurrence through `logcumsumexp` and
never forms `A_t` at all.

This is not an extreme-sequence concern — it bites inside the window lengths
this study searches, so it is measured here rather than assumed.

In [ ]:
from src.minrnn import MinLSTMLayer

print(f"{'T':>6}{'min cumprod(f1)':>18}{'vanilla grad':>16}{'log-space grad':>18}")
for T in [10, 20, 40, 80, 160]:
    torch.manual_seed(0)
    row = []
    for log_space in (False, True):
        layer = MinLSTMLayer(2, 32, log_space=log_space).to(DEVICE)
        x = torch.randn(4, T, 2, device=DEVICE, requires_grad=True)
        layer.forward_parallel(x).sum().backward()
        row.append("finite" if torch.isfinite(x.grad).all() else "NaN")
        if not log_space:
            f1, _ = layer.normalised_gates(x)
            cp = torch.cumprod(f1, dim=1).min().item()
    print(f"{T:>6}{cp:>18.2e}{row[0]:>16}{row[1]:>18}")

## 6. Hyperparameter search

Optuna's TPE sampler with median pruning, **equal trial budget per
architecture** and the same sampler seed. An unequal budget is the quietest way
to rig an architecture comparison, so the budget is a constant.

The window length `W` is searched alongside the network hyperparameters. It
changes the dataset rather than the model, so datasets are cached per `W`.

Two deliberate inclusions:

- **`num_layers=1`.** A single minLSTM layer has gates that depend only on
  `x_t`, making them time-*independent*; it is stacking that gives layer two
  time-dependent inputs. The paper measures 37.6% vs 85.7% on selective copy
  going from one layer to two. Reproducing that here is a free ablation and the
  direct diagnostic for minLSTM's stability.
- **`log_space` is *not* searched.** It is fixed on. The vanilla scan divides
  by `cumprod(f')`, which is ~1e-10 by 20 steps and NaN by 80 — inside the
  window range searched here — so the comparison would be against a model with
  no usable gradients. §6a measures that directly instead of hiding it in the
  search space.

Objective is validation MAE **in metres** on held-out trajectories — not
normalised MSE, so the number is directly comparable with the classical bar.

In [ ]:
import optuna
from src.train import set_seed, train_model, predict, mae_metres
optuna.logging.set_verbosity(optuna.logging.WARNING)

_cache = {}
def datasets_for(W):
    """Window the corpus at a given W, fitting scales on training data only."""
    if W not in _cache:
        Xtr, ytr, _, sc = build_dataset(train_c, W)
        Xva, yva, _, _  = build_dataset(val_c,  W, scales=sc)
        Xte, yte, _, _  = build_dataset(test_c, W, scales=sc)
        _cache[W] = ((Xtr, ytr), (Xva, yva), (Xte, yte), sc)
    return _cache[W]


def evaluate_config(arch, params, seed, return_model=False):
    """Train one configuration and return its validation MAE in metres."""
    train_d, val_d, test_d, sc = datasets_for(params["window"])
    set_seed(seed)
    name = arch
    model = build_model(name, hidden_size=params["hidden"],
                        num_layers=params["layers"], dropout=params["dropout"])
    train_model(model, train_d, val_d, epochs=CFG["max_epochs"],
                batch_size=params["batch"], lr=params["lr"],
                weight_decay=params["weight_decay"], device=DEVICE,
                patience=CFG["patience"])
    val_mae = mae_metres(predict(model, val_d[0], device=DEVICE), val_d[1], sc["y"])
    if return_model:
        test_mae = mae_metres(predict(model, test_d[0], device=DEVICE), test_d[1], sc["y"])
        return val_mae, test_mae, model
    return val_mae


def suggest(trial, arch):
    params = dict(
        window       = trial.suggest_categorical("window", [5, 10, 20, 40]),
        hidden       = trial.suggest_categorical("hidden", [32, 64, 128, 256]),
        layers       = trial.suggest_int("layers", 1, 4),
        dropout      = trial.suggest_float("dropout", 0.0, 0.3),
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        batch        = trial.suggest_categorical("batch", [32, 64, 128, 256]),
        weight_decay = trial.suggest_categorical("weight_decay", [0.0, 1e-5, 1e-4]),
    )
    return params

In [ ]:
studies = {}
for arch in ["lstm", "minlstm"]:
    t0 = time.time()
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=CFG["seed"]),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    )
    study.optimize(lambda t: evaluate_config(arch, suggest(t, arch), seed=CFG["seed"]),
                   n_trials=CFG["n_trials"], show_progress_bar=False)
    studies[arch] = study
    print(f"{arch:<10} best val MAE {study.best_value:.3f} m "
          f"after {len(study.trials)} trials ({time.time()-t0:.0f}s)")
    print(f"           {study.best_params}\n")

In [ ]:
# Does depth matter the way the paper predicts? Plot every trial's outcome by
# layer count -- 1-layer minLSTM should be visibly worse.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (arch, study) in zip(axes, studies.items()):
    done = [t for t in study.trials if t.value is not None]
    layers = [t.params["layers"] for t in done]
    values = [t.value for t in done]
    ax.scatter(layers, values, color=SERIES[arch], alpha=0.65, s=34,
               edgecolor="white", linewidth=0.8)
    for L in sorted(set(layers)):
        best = min(v for l, v in zip(layers, values) if l == L)
        ax.annotate(f"{best:.3f}", (L, best), textcoords="offset points",
                    xytext=(0, -15), ha="center", fontsize=8, color=MUTED)
    ax.axhline(min(CLASSICAL_BAR, CLASSICAL_LS), color=MUTED, ls="--", lw=1.2)
    ax.set_title(LABELS[arch]); ax.set_xlabel("num_layers"); ax.set_xticks([1, 2, 3, 4])
axes[0].set_ylabel("validation MAE (m)")
axes[0].text(0.02, min(CLASSICAL_BAR, CLASSICAL_LS), " tuned classical bar",
             va="bottom", fontsize=8, color=MUTED, transform=axes[0].get_yaxis_transform())
fig.suptitle("Search outcomes by depth", y=1.02)
plt.savefig(RESULTS / "search_by_depth.png"); plt.show()

## 7. Multi-seed confirmation

A single run of the winning configuration proves nothing — the differences here
are small enough to be seed noise, and minLSTM in particular is the less stable
of the two minimal variants (the paper measures 85.7 ± 5.8 for minLSTM against
96.8 ± 3.2 for minGRU at two layers, attributing it to the forget and input
gates being pulled in opposite directions during training).

So the best configuration is re-run across seeds and reported as mean ± std.
**The spread matters as much as the mean here**: a model that is excellent on
three seeds and broken on two is not deployable in a base station.

In [ ]:
final = {}
for arch, study in studies.items():
    params = dict(study.best_params)
    val_maes, test_maes = [], []
    best_model, best_val = None, float("inf")
    for seed in range(CFG["n_seeds"]):
        v, t, model = evaluate_config(arch, params, seed=seed, return_model=True)
        val_maes.append(v); test_maes.append(t)
        if v < best_val:
            best_val, best_model = v, model
    final[arch] = dict(params=params, val=val_maes, test=test_maes,
                       n_params=sum(p.numel() for p in best_model.parameters()))
    torch.save(best_model.state_dict(), MODELS / f"{arch}_best.pt")
    print(f"{arch:<10} test MAE {np.mean(test_maes):.3f} +/- {np.std(test_maes):.3f} m "
          f"over {CFG['n_seeds']} seeds   (seeds: {', '.join(f'{m:.3f}' for m in test_maes)})")

## 8. Generalisation to unseen mobility

The zigzag failure in the original work was a single anecdote. Here it becomes a
controlled axis: train on two mobility regimes, test on the third, which the
model has never seen. Classical baselines are regime-agnostic by construction,
so they provide the reference that a learned model has to justify itself
against *off* its training distribution.

In [ ]:
held_out_rows = []
for regime in REGIMES:
    seen, unseen = split_by_regime(corpus, held_out=regime)
    if not unseen or len(seen) < 10:
        continue
    seen_tr, seen_va, _ = split_by_trajectory(seen, (0.8, 0.2, 0.0),
                                              rng=np.random.default_rng(CFG["seed"]))
    for arch in final:
        p = final[arch]["params"]
        W = p["window"]
        Xtr, ytr, _, sc = build_dataset(seen_tr, W)
        Xva, yva, _, _  = build_dataset(seen_va, W, scales=sc)
        Xun, yun, _, _  = build_dataset(unseen,  W, scales=sc)
        set_seed(CFG["seed"])
        name = arch
        model = build_model(name, hidden_size=p["hidden"], num_layers=p["layers"],
                            dropout=p["dropout"])
        train_model(model, (Xtr, ytr), (Xva, yva), epochs=CFG["max_epochs"],
                    batch_size=p["batch"], lr=p["lr"], device=DEVICE,
                    patience=CFG["patience"])
        held_out_rows.append(dict(
            regime=regime, arch=arch,
            mae=mae_metres(predict(model, Xun, device=DEVICE), yun, sc["y"]),
            kalman=baseline_mae(unseen, W, kalman_cv(dt=CFG["dt"], sigma=CFG["sigma"], q=Q_STAR)),
        ))
        r = held_out_rows[-1]
        print(f"held out {regime:<12} {arch:<10} {r['mae']:.3f} m "
              f"(kalman on same data: {r['kalman']:.3f} m)")

## 9. Results

In [ ]:
rows = [("tuned Kalman (q=%g, W=%d)" % (Q_STAR, W_STAR), CLASSICAL_BAR, None, 0),
        ("tuned LS linear (W=%d)" % W_STAR,              CLASSICAL_LS,  None, 0)]
for arch in final:
    rows.append((LABELS[arch], float(np.mean(final[arch]["test"])),
                 float(np.std(final[arch]["test"])), final[arch]["n_params"]))

print(f"{'model':<30}{'test MAE (m)':>16}{'params':>12}")
print("-" * 58)
for name, mean, std, npar in rows:
    spread = f" ± {std:.3f}" if std is not None else "        "
    par = f"{npar:,}" if npar else "--"
    print(f"{name:<30}{mean:>9.3f}{spread}{par:>12}")

best_learned = min(r[1] for r in rows if r[3])
verdict = ("beats" if best_learned < min(CLASSICAL_BAR, CLASSICAL_LS) else "does NOT beat")
print(f"\nBest learned model {verdict} the tuned classical bar "
      f"({best_learned:.3f} vs {min(CLASSICAL_BAR, CLASSICAL_LS):.3f} m).")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
names  = [r[0] for r in rows]
means  = [r[1] for r in rows]
errs   = [r[2] if r[2] is not None else 0.0 for r in rows]
colours = [MUTED if not r[3] else SERIES.get("lstm" if "LSTM (cuDNN" in r[0] else "minlstm")
           for r in rows]

bars = ax.barh(names, means, xerr=errs, color=colours, height=0.6,
               error_kw=dict(ecolor="#8a8984", lw=1.4, capsize=3))
for bar, mean in zip(bars, means):
    ax.text(bar.get_width() + max(means) * 0.015, bar.get_y() + bar.get_height() / 2,
            f"{mean:.3f}", va="center", fontsize=9, color=INK)
ax.axvline(min(CLASSICAL_BAR, CLASSICAL_LS), color=MUTED, ls="--", lw=1.2)
ax.invert_yaxis()
ax.set_xlabel("test MAE (m) — lower is better")
ax.set_title("Next-position prediction, held-out trajectories")
ax.grid(axis="y", visible=False)
plt.savefig(RESULTS / "final_comparison.png"); plt.show()

In [ ]:
summary = dict(
    config=CFG, quick=QUICK, device=str(DEVICE),
    classical=dict(kalman=CLASSICAL_BAR, ls_linear=CLASSICAL_LS, W=W_STAR, q=Q_STAR),
    models={a: {k: v for k, v in d.items()} for a, d in final.items()},
    timing={a: timing[a]["ms"] for a in ARCHS}, seq_lens=SEQ_LENS,
    held_out=held_out_rows,
)
(RESULTS / "minlstm_study.json").write_text(json.dumps(summary, indent=2, default=float))
print(f"saved -> {RESULTS / 'minlstm_study.json'}")
print(f"saved -> {MODELS}/<arch>_best.pt")

## 10. Reading the results

Three findings to extract, and the traps attached to each:

**Is a learned model worth it?** Compare the final table against the tuned
classical bar. If the networks do not clear it, that is the finding — report it
rather than tuning the baseline down. It would mean the motion model is still
too predictable, and the next move is harder kinematics (stop-and-go,
multi-modal intersection choices, blockage gaps), not a bigger network.

**Is minLSTM faster?** Only past the crossover in §5. At W=10–40, which is where
the accuracy search lands, expect minLSTM to be *slower* than fused cuDNN. State
the crossover sequence length explicitly; a bare "minLSTM is faster" is not
supportable from this data, and neither is "minLSTM is slower."

**Is minLSTM better?** Look at the spread in §7 before the mean. If minLSTM's
seed std is large while the LSTM's is tight, that reproduces the paper's own
observation about minLSTM's two gates and is worth reporting as such — it is
also the point at which adding minGRU (one fewer `Linear` than minLSTM, roughly
five lines in `src/minrnn.py`) becomes the obvious next experiment.

### Next

`6_beamforming_eval.ipynb` carries the winning predictors into the beamforming
evaluation: the digital-twin lookup leak fixed, an Nt = 8…128 array sweep, and
beam misalignment normalised by half-power beamwidth. The predictor is
position-only, so those array sizes cost nothing extra at training time.